# Semester 1 — Fundamentos LLM: Práctica

**Objetivo:** Internalizar los conceptos de tokens, ventana de contexto, temperatura y coste tocando el API directamente. Sin frameworks intermedios.

Estas prácticas corresponden a las lecciones 1.1 a 1.7. Cada ejercicio está diseñado para que el modelo mental se ancle en comportamiento real observable, no en definiciones.

**Prerequisito:** Variable de entorno `OPENAI_API_KEY` configurada.

## Setup: instalación de dependencias

In [ ]:
!pip install openai tiktoken --quiet

## Ejercicio 1 — Llamada básica y tokens consumidos

**Qué practicamos:** Hacer una llamada al API y leer el campo `usage` de la respuesta.

**Por qué importa arquitectónicamente:** Cada respuesta del LLM incluye contabilidad exacta de tokens. Este campo es la base para cualquier sistema de control de costes, rate limiting, y monitoreo de producción. Un arquitecto que no lo lee desde el día uno construye sistemas ciegos.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Eres un asistente conciso. Responde en una frase."},
        {"role": "user",   "content": "¿Qué es un token en el contexto de un LLM?"}
    ]
)

print("Respuesta:", response.choices[0].message.content)
print()
print("--- Contabilidad de tokens ---")
print(f"  prompt_tokens:     {response.usage.prompt_tokens}")
print(f"  completion_tokens: {response.usage.completion_tokens}")
print(f"  total_tokens:      {response.usage.total_tokens}")

## Ejercicio 2 — Temperatura y variabilidad del output

**Qué practicamos:** Ejecutar el mismo prompt con `temperature=0` y con `temperature=1.0` cinco veces cada uno y comparar las respuestas.

**Por qué importa arquitectónicamente:** La temperatura no es un ajuste estético. Es la palanca que controla si tu sistema es reproducible (auditable, testeable) o exploratorio. En producción, la mayoría de pipelines deterministas usan `temperature=0`. Entender esto evita bugs sutiles donde dos ejecuciones del mismo sistema producen resultados distintos sin que nadie lo espere.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

PROMPT = "En una palabra, ¿cuál es la capital de Francia?"
N = 5

def run_n_times(temperature: float, n: int) -> list[str]:
    results = []
    for _ in range(n):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": PROMPT}],
            temperature=temperature,
            max_tokens=10
        )
        results.append(r.choices[0].message.content.strip())
    return results

results_cold = run_n_times(temperature=0.0, n=N)
results_hot  = run_n_times(temperature=1.0, n=N)

print(f"temperature=0.0  → {results_cold}")
print(f"temperature=1.0  → {results_hot}")
print()
print(f"Respuestas únicas (temp=0.0): {len(set(results_cold))}")
print(f"Respuestas únicas (temp=1.0): {len(set(results_hot))}")

## Ejercicio 3 — Contar tokens antes de enviar

**Qué practicamos:** Usar `tiktoken` para contar tokens de un texto localmente, sin hacer ninguna llamada al API.

**Por qué importa arquitectónicamente:** En sistemas RAG y agentes, el contexto se construye dinámicamente. Si no cuentas tokens antes de enviar puedes superar el límite de la ventana de contexto en producción. El conteo local con `tiktoken` es gratis (sin latencia, sin coste). Es el guardián que debe vivir en tu pipeline antes de cada llamada LLM.

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

texts = [
    "Hello",
    "El contexto es el recurso más escaso de un sistema LLM.",
    "\n".join([f"Línea {i}: información de relleno para simular un documento largo." for i in range(50)])
]

MODEL = "gpt-4o-mini"
CONTEXT_LIMIT = 128_000

for text in texts:
    n = count_tokens(text, MODEL)
    pct = n / CONTEXT_LIMIT * 100
    print(f"  {n:6d} tokens | {pct:.2f}% de la ventana | '{text[:50]}...' " if len(text) > 50 else f"  {n:6d} tokens | {pct:.2f}% de la ventana | '{text}'")

## Ejercicio 4 — Calcular el coste real de una conversación

**Qué practicamos:** Multiplicar tokens consumidos por el precio del modelo para obtener coste real en dólares.

**Por qué importa arquitectónicamente:** El coste de un sistema LLM es una función directa de tokens × precio × volumen. Un sistema que procesa 10.000 peticiones/día con un context de 2.000 tokens puede ser barato o arruinoso dependiendo del modelo elegido. Esta práctica te da el instinto para hacer estimaciones rápidas antes de comprometerte con una arquitectura.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Precios en USD por 1M tokens (actualizar según pricing actual de OpenAI)
PRICING = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o":      {"input": 2.50, "output": 10.00},
}

def cost_usd(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    p = PRICING[model]
    return (prompt_tokens * p["input"] + completion_tokens * p["output"]) / 1_000_000

# Simular una conversación de 5 turnos
conversation = [
    {"role": "system", "content": "Eres un asistente de arquitectura de software experto en IA."},
    {"role": "user",   "content": "¿Cuándo debo usar RAG en vez de fine-tuning?"},
]

model = "gpt-4o-mini"
total_cost = 0.0

for turn in range(3):
    response = client.chat.completions.create(
        model=model,
        messages=conversation,
        max_tokens=150
    )
    assistant_msg = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": assistant_msg})

    turn_cost = cost_usd(model, response.usage.prompt_tokens, response.usage.completion_tokens)
    total_cost += turn_cost

    print(f"Turno {turn+1}: {response.usage.prompt_tokens} prompt + {response.usage.completion_tokens} completion tokens → ${turn_cost:.6f}")

    # Siguiente pregunta de usuario (simulada)
    follow_ups = [
        "¿Y si los datos cambian frecuentemente?",
        "¿Qué pasa con el coste de inferencia?"
    ]
    if turn < len(follow_ups):
        conversation.append({"role": "user", "content": follow_ups[turn]})

print()
print(f"Coste total de la conversación: ${total_cost:.6f}")
print(f"Proyección a 10.000 conversaciones/día: ${total_cost * 10_000:.2f}/día")